# 11. Frozen Judge Stability Study

Three total judge repetitions on the same saved outputs and exact inputs. No generation or classifier calls. For the alignment-only pilot, use assessment prompt 0.7.0, SOURCE_STUDY pointing to the existing frozen playground, and STUDY_DOCUMENT_IDS=('3',). This requires nine fresh calls across G1/G2/G3. Old numerical quality outputs below are historical, not alignment results; rerun local analysis for the selected study. Remaining-test must remain separate. Model aliases cannot guarantee an unchanged provider backend over time.

Freeze locally first, inspect the projected calls, then explicitly approve execution. Judgments are development measurements, not independent experts. Repeated calls/criteria do not increase the number of documents. Do not extend repetitions until a preferred ranking appears.

In [ ]:
import json
import os
from pathlib import Path
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from reflection_assessment_feedback.judge_study import (
    freeze_judge_study, load_judge_study, study_progress, run_judge_study, study_results,
)
from reflection_assessment_feedback.judge_stability import analyze_judge_study
from reflection_assessment_feedback.rubric_review import (
    create_rubric_review, rubric_review_cases, validate_rubric_review,
    save_rubric_review_revision,
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != 'prebi_v1':
    raise RuntimeError('Run from the prebi_v1 project root.')
load_dotenv(PROJECT_ROOT / '.env', override=False)
if not os.getenv('PREBI_DATA_ROOT'):
    raise RuntimeError('Set PREBI_DATA_ROOT to approved protected storage.')
PROTECTED_ROOT = Path(os.environ['PREBI_DATA_ROOT']).expanduser().resolve()
STUDY_COHORT = 'playground'
STUDY_CONDITIONS = ('G1', 'G2', 'G3')
STUDY_TASKS = ('assessment_quality',)
ASSESSMENT_PROMPT_VERSION = '0.6.0'
JUDGE_REPETITIONS = 3
IMPORT_EXISTING = False
FREEZE_STUDY = False
SOURCE_STUDY = None  # Existing frozen playground directory for exact-input reuse.
STUDY_DOCUMENT_IDS = None  # Use ('3',) for the single-document pilot.
STUDY_DIRECTORY = None  # Set to an existing frozen study path to resume/analyze.
APPROVE_EXTERNAL_PROCESSING = False
APPROVE_FULL_BATCH = False


## 1. Freeze and inspect locally

The default is a fresh assessment-only playground calibration with prompt `0.6.0`: four documents x three conditions x three judge repetitions = 36 calls. No previous judgments are imported. Keep approvals false until local inspection. Review criterion rationales and rubric overreach, not just condition rankings. Any further prompt edit requires another version and freeze. Lock the calibrated version before a separate remaining-test study; never pool playground results with test results.

Set `FREEZE_STUDY = True` to create an immutable manifest in protected storage, or set `STUDY_DIRECTORY` to resume a previous freeze. Save the printed directory for future sessions. Repeating the same freeze is idempotent. For the original configured prompt use `ASSESSMENT_PROMPT_VERSION = None`; the global experiment configuration is deliberately unchanged so saved generation fingerprints stay valid. Old/new generation or judge versions are never pooled.

The manifest embeds exact packets, source generation files, prompt text/rendered prompts, configuration, imported records/hashes and the shuffled execution schedule. Later prediction/source changes do not reconstruct these inputs. The frozen study, not current notebook selection variables, controls execution.

In [ ]:
if FREEZE_STUDY:
    STUDY_DIRECTORY = freeze_judge_study(
        PROTECTED_ROOT, cohort=STUDY_COHORT, conditions=tuple(STUDY_CONDITIONS),
        judge_repetitions=JUDGE_REPETITIONS, import_existing=IMPORT_EXISTING,
        tasks=tuple(STUDY_TASKS), assessment_prompt_version=ASSESSMENT_PROMPT_VERSION,
        source_study=SOURCE_STUDY, document_ids=STUDY_DOCUMENT_IDS,
    )
if STUDY_DIRECTORY is None:
    raise RuntimeError('Set FREEZE_STUDY=True or supply an existing STUDY_DIRECTORY first.')
STUDY_DIRECTORY = Path(STUDY_DIRECTORY).expanduser().resolve()
frozen = load_judge_study(STUDY_DIRECTORY)
print('Frozen study:', str(STUDY_DIRECTORY))
print(study_progress(STUDY_DIRECTORY))
display(pd.DataFrame([e['expected_metadata'] for e in frozen['protocol']['entries']])[
    ['evaluation_document_id', 'task', 'prompt_id', 'prompt_version']
].drop_duplicates())


## 2. Execute only after approval

This is the only provider-execution cell. Requires approval for processing and the entire frozen study, including its tracing settings. One fresh call per missing packet/repetition; no previous answers are sent. Imported repetition-1 answers do not make calls. Resume skips validated results. Failures stop explicitly: rerun to retry the same slot, never count a failed attempt as a repetition. Calls use the shared judge rate limiter and an exclusive study lock.

The default revised assessment-only playground study requires 36 fresh calls. For a revised assessment-only remaining-test study: 18 documents x 3 conditions x 3 repetitions = 162 fresh calls. Adding the unchanged feedback task gives 324 calls with imports disabled. The historical two-task test protocol had 108 eligible imports and 216 new calls; those counts do not apply to a revised-prompt study. Always use the actual frozen preflight counts. Do not enable approvals merely to analyze saved results; skip this cell.

In [ ]:
if APPROVE_EXTERNAL_PROCESSING and APPROVE_FULL_BATCH:
    print(run_judge_study(
        STUDY_DIRECTORY, approve_external_processing=APPROVE_EXTERNAL_PROCESSING,
        approve_full_batch=APPROVE_FULL_BATCH,
    ))
else:
    print('Execution skipped. Both approvals are required; analysis remains local-only.')


## 3. Coverage and criterion repeatability

Missing judgments and unable-to-judge are distinct. All-equal/varying/extreme proportions use only packets scored in every planned repetition; expected/observed counts and denominators are shown. Pairwise exact agreement uses only the two available scored repetitions and reports its denominator. Distribution tables include zero counts for all 1/2/3 ratings. No pooled assessment/feedback quality score is constructed. Inspect criterion rationales below for evaluator-only reference demands, invented theory/completeness requirements, half-point reasoning and dimension-specific faults. Stability alone is not validity. Keep executed outputs in protected storage, not in version control.

In [ ]:
tables = analyze_judge_study(STUDY_DIRECTORY)
display(tables['coverage'])
ALIGNMENT_STUDY = 'dimension_decisions' in tables
display(tables['dimension_stability'] if ALIGNMENT_STUDY else tables['criterion_stability'])
display(tables['verdict_distribution'] if ALIGNMENT_STUDY else tables['rating_distribution'])
display(tables['pairwise_agreement'])
display(tables['dimension_decisions'] if ALIGNMENT_STUDY else tables['packet_stability'])
review_study, saved_results = study_results(STUDY_DIRECTORY)
identities = {e['packet']['packet_id']: e['identity'] for e in review_study['protocol']['entries']}
rationale_rows = []
for (packet_id, repetition), saved in saved_results.items():
    judgment = saved['judgment']
    identity = identities[packet_id]
    for rating in judgment['result']['dimensions' if ALIGNMENT_STUDY else 'ratings']:
        rationale_rows.append({
            'document_id': identity['document_id'], 'condition': identity['condition'],
            'judge_repetition': repetition, 'task': judgment['task'],
            'prompt_version': judgment['prompt_version'], **rating,
        })
display(pd.DataFrame(rationale_rows))


## 4. Paired condition ordering

Each comparison matches document, generation repetition, task and criterion across all selected conditions. `direction`: 1 means left higher, 0 tie, -1 right higher. A strict reversal requires both directions; a tie transition is reported separately. Stability requires every planned judge repetition. Matched sets can differ across repetitions: inspect the document-level paired table before interpreting aggregate counts. Medians are secondary and require all scored repetitions for every selected condition. No significance tests or independence assumptions.

In [ ]:
if ALIGNMENT_STUDY:
    print('Categorical dimension verdicts are not numerical quality rankings. Inspect dimension decisions above.')
else:
    display(tables['paired_ratings'])
    display(tables['ordinal_counts'])
    display(tables['ordering_stability'])
    display(tables['paired_medians'])


## 5. Human rubric-overreach review

Create a protected worksheet for every assessment dimension, then review frozen assessments, synthesis, reflection evidence and rubric. For the primary study there are 54 assessments / 162 dimension reviews. Judge rationales may suggest cases but are not authoritative.

Issues: `theory_required`, `all_facets_required`, `cross_dimension_chain`, `extra_completeness`, `half_point_explanation`, `other`. Classify each finding as `supported_requirement`, `development_suggestion`, `unsupported_scoring_requirement`, or `ambiguous`. Preserve ambiguity and score impact (`true`/`false`/`null`). A suggestion for more theory is not automatically a scoring error.

Edit the protected worksheet JSON locally. Every finding needs an exact `assessment_excerpt`, exact `rubric_clause`, `affects_score`, valid `evidence_segment_ids`, and `rationale`. Every reviewed case, including no-issue cases, needs `reviewer`, `notes`, and `status='reviewed'`. Do not finalize unresolved issues as confirmed errors. Revisions are append-only, never change the frozen study or prompts. Do not commit worksheet content or notebook output displaying original text.

In [ ]:
CREATE_REVIEW_WORKSHEET = False
if CREATE_REVIEW_WORKSHEET:
    print('Protected worksheet:', create_rubric_review(STUDY_DIRECTORY))
cases = rubric_review_cases(STUDY_DIRECTORY)
display(pd.DataFrame([{k: case[k] for k in ('case_id', 'document_id', 'condition', 'dimension_id')} for case in cases]))
REVIEW_CASE_ID = None  # Set one case ID to inspect its frozen source text locally.
if REVIEW_CASE_ID is not None:
    display(next(case for case in cases if case['case_id'] == REVIEW_CASE_ID))
SAVE_REVIEW_REVISION = False
if SAVE_REVIEW_REVISION:
    worksheet = json.loads((STUDY_DIRECTORY / 'rubric-review' / 'worksheet.json').read_text())
    reviewed = validate_rubric_review(STUDY_DIRECTORY, worksheet)
    print('Reviewed dimensions:', sum(r.status == 'reviewed' for r in reviewed), '/', len(reviewed))
    print('Saved revision:', save_rubric_review_revision(STUDY_DIRECTORY, worksheet))
